In [1]:
%%writefile sales_data.csv
user_id,name,age,category,region,sales,status
1,Amit,25,Electronics,West,500,Active
2,Riya,30,Furniture,East,700,
3,Rohan,17,Electronics,West,300,Active
4,Neha,35,Clothing,North,200,Inactive
5,Amit,25,Electronics,West,500,Active
6,Karan,40,Furniture,East,900,
7,Priya,28,Clothing,West,400,Active
8,Rahul,22,Electronics,South,650,
9,Simran,31,Furniture,North,800,Active
10,Vikas,19,Clothing,West,250,Inactive

Writing sales_data.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark=SparkSession.builder.appName("Week-6 SparkAssignment").getOrCreate()

In [ ]:
df=spark.read.csv(
    "data/sales_data.csv",
    header=True,
    inferSchema=True
)
df.show()

In [ ]:
df.printSchema()
df.columns
df=df.dropDuplicates()
df=df.fillna({
    "status": "Unknown"
})
df.show()

In [ ]:
df.filter(col("age")>=18).show()
df.filter((col("region")=="West") & (col("sales")>300)).show()

In [ ]:
df=df.withColumnRenamed(
    "sales",
    "total_sales"
)
df=df.withColumn(
    "tax",
    col("total_sales") * 0.18
)
df.show()

In [ ]:
df=df.withColumn(
    "age",
    col("age").cast("integer")
)
df.printSchema()

In [ ]:
df.select(
    count("*").alias("total_rows"),
    avg("total_sales").alias("average_sales"),
    min("total_sales").alias("minimum_sales"),
    max("total_sales").alias("maximum_sales")
).show()

In [ ]:
category_df=df.groupBy(
    "category"
).agg(
    sum("total_sales").alias("sales_sum"),
    avg("total_sales").alias("sales_avg"),
    count("*").alias("count")
)
category_df.show()

In [ ]:
df.groupBy("region") \
.agg(sum("total_sales").alias("region_sales")).show()

In [ ]:
spark.read.parquet("output/result_parquet").show()

In [ ]:
result = (
    spark.read.csv("data/sales_data.csv",header=True,inferSchema=True)
    .dropDuplicates()
    .fillna({"status":"Unknown"})
    .filter(col("age") >= 18)
    .withColumnRenamed("sales","total_sales")
    .withColumn("tax", col("total_sales") * 0.18)
    .groupBy("category")
    .agg(
    sum("total_sales").alias("total_sales")
    )
)
result.show()